## 09 — Published Panel Comparison

Evaluate published AISNP panels (Shi 2019, Cao 2022, Cai 2024) on the same
504-sample CN/JPT/SEA task using identical 5-fold CV pipeline as Stage 2 of `08`.

Also checks SNP overlap between our committed panels and each published panel.

In [1]:
import os, sys, json, subprocess
import urllib.request
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Locate project root dynamically
_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts" / "config.py").exists())
sys.path.insert(0, str(_root / "scripts"))
from notebook_init import setup
_cfg, PATHS, POPULATIONS, HARD_FILTERS, SITUATIONAL_FILTERS, ML = setup()

PANELS_DIR  = _root / "data" / "published_panels"
OUTPUT_DIR  = PATHS.outputs_dir("self_evaluation/09_published_panel_comparison")
OUR_PANELS  = PATHS.outputs_dir("self_evaluation/08_unified_panel_sweep") / "panels"
MAF_PLINK   = str(PATHS.outputs_dir("01_hard_filtering") / "SEA_JPT_CN_MAF_filtered")
PSAM        = str(PATHS.outputs_dir("02_situational_filtering") / "SEA_JPT_CN_LD_pruned.psam")
GENO_RAW    = str(OUTPUT_DIR / "published_panels_geno.raw")
COORD_CACHE = str(OUTPUT_DIR / "rsid_coords.json")

os.makedirs(str(OUTPUT_DIR), exist_ok=True)
print(f"Output: {OUTPUT_DIR}")
print(f"Panels: {PANELS_DIR}")


Project root : /home/ibmelab/Projects/AISNP_Research
genomes_data : /mnt/data/aisnp_data/1000genomes/
output root  : output
Output: /mnt/data/aisnp_data/1000genomes/outputs/self_evaluation/09_published_panel_comparison
Panels: /home/ibmelab/Projects/AISNP_Research/data/published_panels


In [2]:
# Population labels from PSAM  (format: #IID  pop)
pop_labels = {}
with open(PSAM) as f:
    for line in f:
        if line.startswith("#"): continue
        parts = line.strip().split("	")
        iid, pop = parts[0], parts[1]   # IID=col0, pop=col1
        pop_labels[iid] = pop

print(f"Samples with labels: {len(pop_labels)}")
print("Populations:", {v: list(pop_labels.values()).count(v) for v in set(pop_labels.values())})


Samples with labels: 504
Populations: {'SEA': 192, 'JPT': 104, 'CN': 208}


In [3]:
# Load rsID lists for each published panel
def load_rsids(fname):
    p = PANELS_DIR / fname
    return [l.strip() for l in open(p) if l.strip()]

PANELS = {
    'shi_36':  load_rsids('shi_36.txt'),
    'shi_59':  load_rsids('shi_59.txt'),
    'shi_98':  load_rsids('shi_98.txt'),
    'shi_142': load_rsids('shi_142.txt'),
    'cao_19':  load_rsids('cao_19.txt'),
}

# cai_eas34: already has coordinates — load separately
cai_coords = []
with open(str(PANELS_DIR / 'cai_eas34_coords.tsv')) as f:
    for line in f:
        parts = line.strip().split('\t')
        cai_coords.append({'chrom': parts[0], 'pos': parts[1],
                           'rsid': parts[2], 'ref': parts[3], 'alt': parts[4]})

all_rsids = sorted({r for rs in PANELS.values() for r in rs})
print(f'Unique rsIDs to map (shi+cao): {len(all_rsids)}')
print(f'Cai EAS34 entries (with coords): {len(cai_coords)}')


Unique rsIDs to map (shi+cao): 161
Cai EAS34 entries (with coords): 34


In [4]:
# Query MyVariant.info for chr:pos (b37) — cached
if os.path.exists(COORD_CACHE):
    with open(COORD_CACHE) as f:
        coord_map = json.load(f)
    print(f'Loaded coord cache: {len(coord_map)} entries')
else:
    print('Querying MyVariant.info...')
    data = json.dumps({
        'q': all_rsids, 'scopes': 'dbsnp.rsid',
        'fields': 'dbsnp.rsid,dbsnp.chrom,dbsnp.hg19.start,dbsnp.ref,dbsnp.alt',
        'size': 1
    }).encode()
    req = urllib.request.Request(
        'https://myvariant.info/v1/query', data=data,
        headers={'Content-Type': 'application/json', 'User-Agent': 'python'})
    results = json.loads(urllib.request.urlopen(req, timeout=60).read())

    coord_map = {}
    for r in results:
        rsid = r.get('query')
        if r.get('notfound') or 'dbsnp' not in r: continue
        db = r['dbsnp']
        coord_map[rsid] = {
            'chrom': str(db.get('chrom','')),
            'pos':   str(db.get('hg19',{}).get('start','')),
            'ref':   db.get('ref',''),
            'alt':   db.get('alt',''),
        }

    # Add cai coords
    for c in cai_coords:
        coord_map[c['rsid']] = {'chrom': c['chrom'], 'pos': c['pos'],
                                'ref': c['ref'], 'alt': c['alt']}

    with open(COORD_CACHE, 'w') as f:
        json.dump(coord_map, f)
    print(f'Mapped {len(coord_map)} rsIDs, cached.')


Loaded coord cache: 182 entries


In [5]:
# Search MAF-filtered pvar for matching positions, extract genotypes (cached)
if os.path.exists(GENO_RAW):
    print(f'Genotype cache exists: {GENO_RAW}')
else:
    print('Searching pvar for published panel positions...')
    pos_set = {(c['chrom'], c['pos']) for c in coord_map.values()}

    # Write positions file for awk
    pos_file = str(OUTPUT_DIR / 'target_positions.tsv')
    with open(pos_file, 'w') as f:
        for chrom, pos in sorted(pos_set):
            f.write(f'{chrom}\t{pos}\n')

    # Find matching pvar lines
    pvar_path = MAF_PLINK + '.pvar'
    matched_ids_file = str(OUTPUT_DIR / 'matched_snp_ids.txt')
    cmd = (f"awk 'NR==FNR{{pos[$1\"_\"$2]=1;next}} /^#/{{next}} "
           f"{{key=$1\"_\"$2; if(key in pos) print $3}}' "
           f"{pos_file} {pvar_path} > {matched_ids_file}")
    subprocess.run(cmd, shell=True, check=True)
    n_matched = sum(1 for _ in open(matched_ids_file))
    print(f'Matched {n_matched} variants in pvar')

    # Extract with plink2
    out_prefix = str(OUTPUT_DIR / 'published_panels_geno')
    result = subprocess.run([
        'plink2', '--pfile', MAF_PLINK, '--keep', PSAM,
        '--extract', matched_ids_file,
        '--export', 'A', '--out', out_prefix,
        '--threads', '8', '--memory', '8000'
    ], capture_output=True, text=True)
    print(result.stdout[-500:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-300:])
    else:
        print(f'Genotype file: {GENO_RAW}')


Genotype cache exists: /mnt/data/aisnp_data/1000genomes/outputs/self_evaluation/09_published_panel_comparison/published_panels_geno.raw


In [6]:
# Parse .raw file into sample × SNP matrix
print('Parsing genotype matrix...')
with open(GENO_RAW) as f:
    header = f.readline().strip().split()
    snp_cols_raw = header[6:]   # FID IID PAT MAT SEX PHENOTYPE then SNPs

    sample_ids = []
    geno_rows  = []
    for line in f:
        parts = line.strip().split()
        iid = parts[1]
        sample_ids.append(iid)
        geno_rows.append([0 if v == 'NA' else int(float(v)) for v in parts[6:]])

G_pub = np.array(geno_rows, dtype=np.float32)
print(f'Genotype matrix: {G_pub.shape}  (samples × SNPs)')

# Population labels aligned to sample order
y_str = np.array([pop_labels[s] for s in sample_ids])
le    = LabelEncoder()
y     = le.fit_transform(y_str)
print(f'Classes: {le.classes_}  counts: {dict(zip(le.classes_, np.bincount(y)))}')

# Build col name → index in G_pub
# raw col names are like "1:40084488:\G:\T_T" → strip counted-allele suffix
col_to_idx = {}
for i, col in enumerate(snp_cols_raw):
    # plink appends _ALLELE, strip it: last underscore onward
    base = col[:col.rfind('_')]
    col_to_idx[base] = i

print(f'Column map entries: {len(col_to_idx)}')


Parsing genotype matrix...
Genotype matrix: (504, 182)  (samples × SNPs)
Classes: ['CN' 'JPT' 'SEA']  counts: {np.str_('CN'): np.int64(208), np.str_('JPT'): np.int64(104), np.str_('SEA'): np.int64(192)}
Column map entries: 182


In [7]:
# Map each published panel rsID → column index in G_pub
# plink ID format: chr:pos:\REF:\ALT  (backslashes)
def rsids_to_idx(rsid_list):
    idxs = []
    for rsid in rsid_list:
        c = coord_map.get(rsid)
        if not c: continue
        pid_fwd = f"{c['chrom']}:{c['pos']}:\\{c['ref']}:\\{c['alt']}"
        pid_rev = f"{c['chrom']}:{c['pos']}:\\{c['alt']}:\\{c['ref']}"
        if pid_fwd in col_to_idx:
            idxs.append(col_to_idx[pid_fwd])
        elif pid_rev in col_to_idx:
            idxs.append(col_to_idx[pid_rev])
    return idxs

panel_idx = {}
for pname, rsid_list in PANELS.items():
    idx = rsids_to_idx(rsid_list)
    panel_idx[pname] = idx
    print(f'  {pname}: {len(idx)}/{len(rsid_list)} SNPs matched')

# cai_eas34 via coords directly
cai_idx = []
for c in cai_coords:
    pid_fwd = f"{c['chrom']}:{c['pos']}:\\{c['ref']}:\\{c['alt']}"
    pid_rev = f"{c['chrom']}:{c['pos']}:\\{c['alt']}:\\{c['ref']}"
    if pid_fwd in col_to_idx: cai_idx.append(col_to_idx[pid_fwd])
    elif pid_rev in col_to_idx: cai_idx.append(col_to_idx[pid_rev])
panel_idx['cai_eas34'] = cai_idx
print(f'  cai_eas34: {len(cai_idx)}/{len(cai_coords)} SNPs matched')


  shi_36: 29/36 SNPs matched
  shi_59: 49/59 SNPs matched
  shi_98: 80/98 SNPs matched
  shi_142: 116/142 SNPs matched
  cao_19: 14/19 SNPs matched
  cai_eas34: 34/34 SNPs matched


In [8]:
# 5-fold CV evaluation for each published panel
CLASSIFIERS = {
    'RF':      (RandomForestClassifier(n_estimators=200, random_state=42), False),
    'LR':      (LogisticRegression(max_iter=2000, solver='saga', random_state=42), True),
    'SVM_RBF': (SVC(kernel='rbf', probability=True, random_state=42), True),
}

from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def eval_panel(X_panel, y, panel_name):
    records = []
    for clf_name, (clf_tmpl, needs_scale) in CLASSIFIERS.items():
        fold_acc, fold_f1, fold_mcc, fold_auc = [], [], [], []
        for tr_idx, te_idx in skf.split(X_panel, y):
            X_tr, X_te = X_panel[tr_idx], X_panel[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]
            if needs_scale:
                sc = StandardScaler().fit(X_tr)
                X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)
            clf = clone(clf_tmpl).fit(X_tr, y_tr)
            y_pred = clf.predict(X_te)
            fold_acc.append(accuracy_score(y_te, y_pred))
            fold_f1.append(f1_score(y_te, y_pred, average='weighted'))
            fold_mcc.append(matthews_corrcoef(y_te, y_pred))
            if hasattr(clf, 'predict_proba'):
                fold_auc.append(roc_auc_score(y_te, clf.predict_proba(X_te),
                                              multi_class='ovr', average='macro'))
        records.append({'panel': panel_name, 'classifier': clf_name,
                        'n_snps': X_panel.shape[1],
                        'acc':  np.mean(fold_acc),  'f1':  np.mean(fold_f1),
                        'mcc':  np.mean(fold_mcc),  'roc_auc': np.mean(fold_auc) if fold_auc else None})
    return records

all_records = []
for pname, idxs in panel_idx.items():
    if not idxs:
        print(f'Skipping {pname}: no matched SNPs')
        continue
    X_p = G_pub[:, idxs]
    print(f'\nEvaluating {pname} ({X_p.shape[1]} SNPs)...')
    recs = eval_panel(X_p, y, pname)
    for r in recs:
        print(f"  {r['classifier']:8s}: acc={r['acc']:.4f}  f1={r['f1']:.4f}  mcc={r['mcc']:.4f}")
    all_records.extend(recs)

import csv
out_csv = str(OUTPUT_DIR / 'published_panel_results.csv')
with open(out_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['panel','classifier','n_snps','acc','f1','mcc','roc_auc'])
    w.writeheader(); w.writerows(all_records)
print(f'\nSaved: {out_csv}')



Evaluating shi_36 (29 SNPs)...
  RF      : acc=0.6806  f1=0.6806  mcc=0.5032
  LR      : acc=0.7005  f1=0.6992  mcc=0.5359
  SVM_RBF : acc=0.6806  f1=0.6755  mcc=0.5077

Evaluating shi_59 (49 SNPs)...
  RF      : acc=0.6846  f1=0.6846  mcc=0.5090
  LR      : acc=0.7084  f1=0.7093  mcc=0.5451
  SVM_RBF : acc=0.7242  f1=0.7223  mcc=0.5725

Evaluating shi_98 (80 SNPs)...
  RF      : acc=0.7084  f1=0.7070  mcc=0.5482
  LR      : acc=0.7342  f1=0.7325  mcc=0.5904
  SVM_RBF : acc=0.7758  f1=0.7744  mcc=0.6541

Evaluating shi_142 (116 SNPs)...
  RF      : acc=0.7421  f1=0.7418  mcc=0.5996
  LR      : acc=0.7678  f1=0.7676  mcc=0.6398
  SVM_RBF : acc=0.7857  f1=0.7850  mcc=0.6680

Evaluating cao_19 (14 SNPs)...
  RF      : acc=0.7877  f1=0.7862  mcc=0.6714
  LR      : acc=0.8194  f1=0.8190  mcc=0.7188
  SVM_RBF : acc=0.8094  f1=0.8081  mcc=0.7033

Evaluating cai_eas34 (34 SNPs)...
  RF      : acc=0.9286  f1=0.9284  mcc=0.8894
  LR      : acc=0.9345  f1=0.9343  mcc=0.8988
  SVM_RBF : acc=0.946

In [9]:
# Best classifier per panel — comparison table
best = {}
for r in all_records:
    k = r['panel']
    if k not in best or r['acc'] > best[k]['acc']:
        best[k] = r

print('=' * 75)
print('PUBLISHED PANEL COMPARISON  (best classifier per panel, 5-fold CV)')
print('=' * 75)
print(f'{"Panel":<12} {"N SNPs":>6}  {"Clf":>8}  {"Acc":>7}  {"F1":>7}  {"MCC":>7}  {"ROC-AUC":>8}')
print('-' * 75)
order = ['shi_36','shi_59','shi_98','shi_142','cao_19','cai_eas34']
for k in order:
    if k not in best: continue
    r = best[k]
    auc = f"{r['roc_auc']:.4f}" if r['roc_auc'] else ' N/A '
    print(f"{k:<12} {r['n_snps']:>6}  {r['classifier']:>8}  {r['acc']:>7.4f}  {r['f1']:>7.4f}  {r['mcc']:>7.4f}  {auc:>8}")

print()
print('For reference — our pipeline Stage 2 CV (stat+EN):')
print(f'  N=35 → acc≈0.9226   N=50 → acc≈0.9305   N=55 → acc≈0.9543')


PUBLISHED PANEL COMPARISON  (best classifier per panel, 5-fold CV)
Panel        N SNPs       Clf      Acc       F1      MCC   ROC-AUC
---------------------------------------------------------------------------
shi_36           29        LR   0.7005   0.6992   0.5359    0.8469
shi_59           49   SVM_RBF   0.7242   0.7223   0.5725    0.8560
shi_98           80   SVM_RBF   0.7758   0.7744   0.6541    0.8952
shi_142         116   SVM_RBF   0.7857   0.7850   0.6680    0.9087
cao_19           14        LR   0.8194   0.8190   0.7188    0.9428
cai_eas34        34   SVM_RBF   0.9464   0.9464   0.9174    0.9909

For reference — our pipeline Stage 2 CV (stat+EN):
  N=35 → acc≈0.9226   N=50 → acc≈0.9305   N=55 → acc≈0.9543


In [10]:
# SNP overlap: our committed panels vs published panels
# Load our panel SNP IDs (stat panel snp_ids are sanitized chr:pos_b37_REF,ALT)

def load_our_panel(n):
    p = OUR_PANELS / f'panel_N{n:03d}.csv'
    if not p.exists(): return set()
    ids = set()
    with open(str(p)) as f:
        next(f)  # header
        for line in f:
            snp_id = line.strip().split(',')[1]
            # de-sanitize: _b37_ → [b37]
            raw_id = snp_id.replace('_b37_', '[b37]', 1)
            ids.add(raw_id)
    return ids

# Published panel SNPs as chr:pos[b37]REF,ALT strings
def pub_as_matrix_ids(rsid_list):
    ids = set()
    for rsid in rsid_list:
        c = coord_map.get(rsid)
        if not c: continue
        ids.add(f"{c['chrom']}:{c['pos']}[b37]{c['ref']},{c['alt']}")
        ids.add(f"{c['chrom']}:{c['pos']}[b37]{c['alt']},{c['ref']}")  # either strand
    return ids

print('SNP OVERLAP — our N=35/50/70 panels vs published')
print('=' * 60)
for our_n in [35, 50, 70]:
    our_ids = load_our_panel(our_n)
    print(f'\nOur N={our_n}: {len(our_ids)} SNPs')
    for pname, rsid_list in list(PANELS.items()) + [('cai_eas34', [c['rsid'] for c in cai_coords])]:
        pub_ids = pub_as_matrix_ids(rsid_list)
        overlap = len(our_ids & pub_ids)
        print(f'  vs {pname:<12}: {overlap} shared / {len(pub_ids)} pub SNPs')


SNP OVERLAP — our N=35/50/70 panels vs published

Our N=35: 35 SNPs
  vs shi_36      : 0 shared / 72 pub SNPs
  vs shi_59      : 0 shared / 118 pub SNPs
  vs shi_98      : 0 shared / 196 pub SNPs
  vs shi_142     : 0 shared / 284 pub SNPs
  vs cao_19      : 0 shared / 38 pub SNPs
  vs cai_eas34   : 0 shared / 68 pub SNPs

Our N=50: 50 SNPs
  vs shi_36      : 0 shared / 72 pub SNPs
  vs shi_59      : 0 shared / 118 pub SNPs
  vs shi_98      : 0 shared / 196 pub SNPs
  vs shi_142     : 0 shared / 284 pub SNPs
  vs cao_19      : 0 shared / 38 pub SNPs
  vs cai_eas34   : 0 shared / 68 pub SNPs

Our N=70: 70 SNPs
  vs shi_36      : 0 shared / 72 pub SNPs
  vs shi_59      : 0 shared / 118 pub SNPs
  vs shi_98      : 0 shared / 196 pub SNPs
  vs shi_142     : 0 shared / 284 pub SNPs
  vs cao_19      : 0 shared / 38 pub SNPs
  vs cai_eas34   : 0 shared / 68 pub SNPs
